# Batch 02: time and record integrity
Read-only verification of the eight reviewed fields. The authoritative Data Contract remains unchanged. The review registry remains pending until the team agrees the field decisions.
Run from the work_place root or a child directory using the project Python environment. Missing stored hours are not inserted. Season counts do not establish 365-day endpoint support.

In [1]:
from pathlib import Path
import csv,hashlib,json
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from data3001.dataset import development_ids,track_path
root=Path.cwd()
if not (root/'configs/dataset_manifest.json').exists():
    candidates=list(root.parents)
    root=next(p for p in candidates if (p/'configs/dataset_manifest.json').exists())
# Paths are resolved relative to the project root; no test records are read.
manifest=json.loads((root/'configs/dataset_manifest.json').read_text())
contract_hash=hashlib.sha256((root/'docs/contract/data_contract_v1.0.xlsx').read_bytes()).hexdigest()
assert contract_hash==manifest['contract_sha256']
fields='time time_UTC departure_day_UTC departure_year season time_ok duplicate_status analytical_row'.split()
nulls={k:0 for k in fields}
checks={k:0 for k in ['invalid_epoch','off_hour_grid','utc_mismatch','day_mismatch','year_mismatch','season_mismatch','time_ok_mismatch','analytical_row_mismatch','duplicate_key_extra_rows','nonpositive_adjacent_intervals','stored_gap_mismatch']}
counts={'IDs':0,'rows':0,'R_rows':0,'time_ok_false':0,'analytical_row_false':0,'duplicate_1':0,'duplicate_2':0,'gap_intervals':0,'absent_hours':0,'IDs_with_gap':0,'R_current_rows_following_gap':0,'R_following_gap_absent_hours':0}
seasons={str(k):{'rows':0,'IDs':set(),'R_rows':0,'R_IDs':set()} for k in range(4)}
minimum=float('inf');maximum=float('-inf');largest_gap=0;example=None
for ident in development_ids():
    path=track_path(ident) # Verifies the frozen per-ID checksum before reading.
    tab=pq.read_table(path,columns=fields+['in_R','absent_hours_before'])
    n=tab.num_rows;assert n==manifest['records'][str(ident)]['rows']
    counts['IDs']+=1;counts['rows']+=n
    for k in fields:nulls[k]+=tab[k].null_count
    t=tab['time'].to_numpy();in_R=tab['in_R'].to_numpy()
    utc=tab['time_UTC'].cast(pa.int64()).to_numpy()
    assert pa.types.is_timestamp(tab['time_UTC'].type) and tab['time_UTC'].type.unit=='ms' and tab['time_UTC'].type.tz=='UTC'
    date=t.astype('datetime64[s]');month=date.astype('datetime64[M]').astype('int64')%12+1
    yr=date.astype('datetime64[Y]').astype('int64')+1970
    expected_season=np.select([np.isin(month,[12,1,2]),np.isin(month,[3,4,5]),np.isin(month,[6,7,8]),np.isin(month,[9,10,11])],[0,1,2,3],default=-1)
    season=tab['season'].to_numpy();tok=tab['time_ok'].to_numpy();duplicate=tab['duplicate_status'].to_numpy();analytical=tab['analytical_row'].to_numpy()
    expected_tok=np.isfinite(t)&(t>=-62135596800)&(t<=253402300799)
    checks['invalid_epoch']+=int((~expected_tok).sum());checks['off_hour_grid']+=int((t%3600!=0).sum())
    checks['utc_mismatch']+=int((utc!=t*1000).sum())
    checks['day_mismatch']+=int((tab['departure_day_UTC'].to_numpy()!=np.floor(t/86400)).sum())
    checks['year_mismatch']+=int((tab['departure_year'].to_numpy()!=yr).sum())
    checks['season_mismatch']+=int((season!=expected_season).sum())
    checks['time_ok_mismatch']+=int((tok!=expected_tok).sum())
    checks['analytical_row_mismatch']+=int((analytical!=((duplicate==0)&tok)).sum())
    checks['duplicate_key_extra_rows']+=int(n-np.unique(t).size)
    counts['time_ok_false']+=int((~tok).sum());counts['analytical_row_false']+=int((~analytical).sum())
    counts['duplicate_1']+=int((duplicate==1).sum());counts['duplicate_2']+=int((duplicate==2).sum())
    counts['R_rows']+=int(in_R.sum())
    dt=np.diff(t);checks['nonpositive_adjacent_intervals']+=int((dt<=0).sum())
    absent=np.full(n,-1,dtype=np.int64);good=(dt>0)&np.isfinite(dt)&(dt%3600==0)
    absent[1:][good]=(dt[good]/3600-1).astype(np.int64)
    checks['stored_gap_mismatch']+=int((absent!=tab['absent_hours_before'].to_numpy()).sum())
    gap=absent>0;counts['gap_intervals']+=int(gap.sum());counts['absent_hours']+=int(absent.clip(0).sum());counts['IDs_with_gap']+=int(gap.any())
    counts['R_current_rows_following_gap']+=int((gap&in_R).sum());counts['R_following_gap_absent_hours']+=int(absent[gap&in_R].sum())
    if gap.any():
        largest_gap=max(largest_gap,int(absent.max()))
        if example is None:
            j=int(np.flatnonzero(gap)[0]);example={'id':ident,'previous_UTC':str(date[j-1]),'current_UTC':str(date[j]),'missing_hours':int(absent[j])}
    for k in range(4):
        mask=season==k;s=seasons[str(k)];s['rows']+=int(mask.sum());s['R_rows']+=int((mask&in_R).sum())
        if mask.any():s['IDs'].add(ident)
        if (mask&in_R).any():s['R_IDs'].add(ident)
    minimum=min(minimum,float(t.min()));maximum=max(maximum,float(t.max()))
for s in seasons.values():s['IDs']=len(s['IDs']);s['R_IDs']=len(s['R_IDs'])
assert not any(nulls.values()) and not any(checks.values())
assert counts['rows']==33943292 and counts['IDs']==3224
result={'scope':'Development pool only; final-test trajectories not opened. All counts refer to stored observations, not independent samples or horizon-qualified endpoints.','dataset_release':manifest['version'],'dataset_manifest_sha256':hashlib.sha256((root/'configs/dataset_manifest.json').read_bytes()).hexdigest(),'contract_sha256':contract_hash,'null_rows_by_field':nulls,'consistency_failures':checks,'counts':counts,'season_codes':{'0':'DJF','1':'MAM','2':'JJA','3':'SON'},'season_support_observations':seasons,'minimum_UTC':str(np.datetime64(int(minimum),'s')),'maximum_UTC':str(np.datetime64(int(maximum),'s')),'largest_gap_missing_hours':largest_gap,'gap_example':example,'input_file_checksums_verified':counts['IDs'],'data_modified':False,'registry_decisions_changed':False}
print(json.dumps(result,indent=2))


{
  "scope": "Development pool only; final-test trajectories not opened. All counts refer to stored observations, not independent samples or horizon-qualified endpoints.",
  "dataset_release": "per-drifter-20261016-v1",
  "dataset_manifest_sha256": "4988b6aee5ca34daa486caf3a43d4526eaa874da3274977331f0283ca3f6694b",
  "contract_sha256": "6cfa91471a1399a9c3a1a8edfeb2e18cea17e0f4031ef0f8ca604ca4565a2446",
  "null_rows_by_field": {
    "time": 0,
    "time_UTC": 0,
    "departure_day_UTC": 0,
    "departure_year": 0,
    "season": 0,
    "time_ok": 0,
    "duplicate_status": 0,
    "analytical_row": 0
  },
  "consistency_failures": {
    "invalid_epoch": 0,
    "off_hour_grid": 0,
    "utc_mismatch": 0,
    "day_mismatch": 0,
    "year_mismatch": 0,
    "season_mismatch": 0,
    "time_ok_mismatch": 0,
    "analytical_row_mismatch": 0,
    "duplicate_key_extra_rows": 0,
    "nonpositive_adjacent_intervals": 0,
    "stored_gap_mismatch": 0
  },
  "counts": {
    "IDs": 3224,
    "rows": 3394